# scVI best - nph_cerebellum

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'nph_cerebellum'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('Sample',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'Celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'nph_cerebellum'
assert BATCH_KEYS == ('Sample',)
assert LABEL_KEY == 'Celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                          nph_cerebellum
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                             [Sample]
label_key                                              Celltype
metrics                [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=84619 n_batches=2 n_labels=8 counts_layer=True
CPCR preflight: eligible_labels=8


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
OPTUNA_BEST_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/nph_cerebellum/expanded_metrics_v1/scvi_nph_cerebellum_joint_expanded_metrics_v1_best_params.json'
OPTUNA_TRIALS_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/nph_cerebellum/expanded_metrics_v1/scvi_nph_cerebellum_joint_expanded_metrics_v1_trials.csv'
OPTUNA_STUDY_NAME = 'scvi_nph_cerebellum_joint_expanded_metrics_v1'
OPTUNA_BEST_TRIAL = 143
OPTUNA_BEST_VALUE = 0.6463484893397522
OPTUNA_BEST_PARAMS = {'n_latent': 46, 'n_hidden': 256, 'n_layers': 3, 'dropout_rate': 0.016126223470503704, 'lr': 0.0008082713114682404}

with open(OPTUNA_BEST_SOURCE, "r", encoding="utf-8") as source_file:
    current_source = json.load(source_file)
assert current_source["study_name"] == OPTUNA_STUDY_NAME
source_dataset_name = current_source.get(
    "dataset_name",
    Path(current_source["dataset"]).stem,
)
assert source_dataset_name == DATASET_NAME
assert current_source["best_value"] == OPTUNA_BEST_VALUE
assert current_source["best_params"] == OPTUNA_BEST_PARAMS
assert tuple(OPTUNA_BEST_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

SCVI_PARAMS = dict(OPTUNA_BEST_PARAMS)

pd.Series(
    {
        "source": OPTUNA_BEST_SOURCE,
        "trials_source": OPTUNA_TRIALS_SOURCE,
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="best_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7193805818862319


optimised clustering against scvi_metric_label
optimal cluster resolution: 0.4
optimal score: 0.8070122605225689
BRAS...


seed           9.000000
BAMI           0.807012
CASW           0.522371
SCGRAPHM       0.631443
BRAS           0.932867
KBET           0.445596
CPCR           0.399164
bio_score      0.653609
batch_score    0.592543
joint_score    0.629182
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.812045,0.529858,0.691665,0.935808,0.468142,0.393311,0.677856,0.599087,0.646348
1,1,0.790365,0.522739,0.655181,0.930759,0.443948,0.346114,0.656095,0.573607,0.623100
2,2,0.821321,0.526872,0.621451,0.930882,0.478393,0.395963,0.656548,0.601746,0.634627
3,3,0.808737,0.524351,0.665439,0.934943,0.483285,0.354334,0.666176,0.590854,0.636047
4,4,0.784926,0.517342,0.648606,0.933551,0.474421,0.347992,0.650291,0.585321,0.624303
5,5,0.768932,0.522317,0.628121,0.937122,0.441951,0.385320,0.639790,0.588131,0.619126
6,6,0.798473,0.525043,0.662960,0.931267,0.463788,0.324814,0.662158,0.573290,0.626611
7,7,0.771018,0.525155,0.621237,0.935771,0.476385,0.332261,0.639137,0.581472,0.616071
8,8,0.788645,0.523408,0.665070,0.930538,0.485296,0.400656,0.659041,0.605497,0.637623
9,9,0.807012,0.522371,0.631443,0.932867,0.445596,0.399164,0.653609,0.592543,0.629182


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.795148,0.016572,0.768932,0.821321
CASW,0.523946,0.003112,0.517342,0.529858
SCGRAPHM,0.649117,0.022036,0.621237,0.691665
BRAS,0.933351,0.002324,0.930538,0.937122
KBET,0.466121,0.015792,0.441951,0.485296
CPCR,0.367993,0.028229,0.324814,0.400656
bio_score,0.656070,0.011019,0.639137,0.677856
batch_score,0.589155,0.010520,0.573290,0.605497
joint_score,0.629304,0.008849,0.616071,0.646348
